# Train LFM3-VL-2B for Wind Turbine Blade Inspection with GRPO and TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/examples/wind-turbine-inspection/notebooks/lfm3_vl_grpo.ipynb)

This notebook continues training the adapter from [`lfm3_vl_sft.ipynb`](lfm3_vl_sft.ipynb) with Group Relative Policy Optimization (GRPO) from [TRL](https://github.com/huggingface/trl). For each blade photo, the model samples four answers. A reward scores each answer's boxes against the annotated defects, and answers that score above their group's average are reinforced.

SFT teaches the model the defect types and the answer format, but it also draws many extra boxes. GRPO optimizes the detection score directly. In our run, 400 GRPO steps raised detection F1 on the validation split from 0.089 to 0.120, mostly by cutting wrong boxes from 1,103 to 710.

## 🎯 What you'll find

- **A detection reward**: soft matching by type, overlap and position, plus the exact IoU ≥ 0.5 score that the evaluation reports, with tests that check it ranks answers sensibly
- **GRPO + LoRA**: continue training the SFT adapter, with a small KL penalty that keeps it close to where it started
- **Monitoring**: detection F1 on 200 fixed validation photos before training and at every checkpoint, plus logs of rewards and sampled answers

**Before you start:** run [`lfm3_vl_sft.ipynb`](lfm3_vl_sft.ipynb) and push its adapter to the Hub.

**Hardware:** a Colab A100. 400 steps take about 31 minutes, plus a few minutes for each validation check.

## Install and authenticate

On Colab, add your Hugging Face token as the `HF_TOKEN` secret (key icon in the sidebar). It needs read access to your SFT adapter and write access to push the result.

In [ ]:
%pip install -q "transformers==5.20.0" "trl==1.16.0" "peft==0.21.2" "torchao==0.18.0" datasets accelerate pillow pandas matplotlib rich

import os
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
assert token, "Add HF_TOKEN to Colab secrets and grant this notebook access."
login(token=token, add_to_git_credential=False)

import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to an A100 (Runtime > Change runtime type)."
print(torch.cuda.get_device_name(0))

## Configuration

Set `SFT_ADAPTER_REPO` to the adapter you pushed from the SFT notebook, and `HUB_REPO` to where this notebook should push its result.

Each optimizer step uses one photo and four sampled answers, so a full pass over the 3,315 training photos would be 3,315 steps. `MAX_STEPS = 400` stops much earlier: in our run, step 400 scored best on the validation split, and step 800 was already worse. Raise `MAX_STEPS` to explore longer runs, and compare the saved checkpoints on the validation split.

In [ ]:
from pathlib import Path
import random
import sys

SEED = 42
MODEL_ID = "LiquidAI/LFM3-VL-2B"
SFT_ADAPTER_REPO = "your-username/lfm3-vl-blade-sft"  # the adapter pushed by lfm3_vl_sft.ipynb
HUB_REPO = None  # e.g. "your-username/lfm3-vl-blade-grpo", to push the trained adapter
OUTPUT_DIR = "./outputs/lfm3-vl-blade-grpo"  # use a new directory for each run
MAX_TRAIN_ROWS = None  # None uses all 3,315 balanced training photos
MAX_STEPS = 400
SAVE_STEPS = 200  # save a checkpoint and score the 200 validation photos this often
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 256
LEARNING_RATE = 1e-5
BOX_FORMAT = "xyxy1000"
assert not Path(OUTPUT_DIR).exists(), f"{OUTPUT_DIR} exists; choose a fresh output directory."

if not Path("blade_eval.py").exists():
    from google.colab import files
    print("Upload blade_eval.py from the example folder")
    files.upload()
assert Path("blade_eval.py").exists()
import blade_eval as be

base_info = be.model_identity(MODEL_ID)
start_info = be.model_identity(SFT_ADAPTER_REPO)
print("Base model:", base_info)
print("Starting SFT adapter:", start_info)


## Training photos and the validation monitor

The training photos are the same balanced set as in the SFT notebook: at most 600 per defect type, with every augmented copy of a photo in the same split. `image` and the conversational `prompt` are the columns TRL expects; extra columns are passed to the reward functions. The 200 validation photos are fixed and never used for training.

In [ ]:
from collections import defaultdict, Counter
import json
from datasets import Dataset, Image as DatasetImage

root = be.download("./data")
train_items = be.load_items(root, split="train", exclude_cross_class_duplicates=True)
val_items = be.load_items(root, split="val", n=200, exclude_cross_class_duplicates=True)
assert len(train_items) == 4915 and len(val_items) == 200
rng = random.Random(SEED)
by_class = defaultdict(list)
for item in train_items:
    by_class[item.label].append(item)
balanced = [item for items in by_class.values() for item in (rng.sample(items, 600) if len(items) > 600 else items)]
rng.shuffle(balanced)
assert len(balanced) == 3315
selected_items = balanced if MAX_TRAIN_ROWS is None else balanced[:MAX_TRAIN_ROWS]
print("Training rows:", len(selected_items), "| class counts:", dict(Counter(item.label for item in selected_items)))

PROMPT = be.build_prompt(BOX_FORMAT)
rows = [{
    "item_id": item.item_id,
    "image": str(item.path),
    "prompt": [{"role": "user", "content": PROMPT}],  # TRL prepends the image block from the image column
    "gt_boxes": json.dumps([[label, coords] for label, coords in item.boxes]),
} for item in selected_items]
train_ds = Dataset.from_list(rows).cast_column("image", DatasetImage())
print(train_ds[0]["item_id"], json.loads(train_ds[0]["gt_boxes"]))

## Reward: type-aware soft matching and exact hits

The primary reward gives each predicted box at most one ground-truth match. It combines IoU with a small center-and-size proximity term, so near misses can score above unrelated boxes even below IoU 0.5. The denominator penalizes missing and duplicate boxes. An exact IoU-0.5 F1 component keeps the reward tied to the reported metric. Formatting is a small separate reward. The tests in the next cell check that the reward ranks answers the way it should. During training, sampled answers and their rewards are saved, so you can check that the model isn't gaming the reward.

In [ ]:
import json
import math
import os


def completion_text(completion):
    if isinstance(completion, str):
        return completion.strip()
    content = completion[-1]["content"]
    if isinstance(content, list):
        content = "".join(part.get("text", "") for part in content)
    return content.strip()


def strict_boxes(text):
    try:
        entries = json.loads(text.strip())
    except (ValueError, TypeError):
        return None
    if not isinstance(entries, list) or len(entries) > 20:
        return None
    boxes = []
    for entry in entries:
        if not isinstance(entry, dict) or "label" not in entry or "bbox_2d" not in entry:
            return None
        label = be.normalize_label(entry["label"])
        coords = entry["bbox_2d"]
        if label not in be.CLASSES or not isinstance(coords, list) or len(coords) != 4:
            return None
        if any(isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) for value in coords):
            return None
        x1, y1, x2, y2 = coords
        if not (0 <= x1 < x2 <= 1000 and 0 <= y1 < y2 <= 1000):
            return None
        boxes.append((label, [value / 1000 for value in coords]))
    return boxes


def pair_quality(pred, target):
    if pred[0] != target[0]:
        return 0.0
    p, g = pred[1], target[1]
    overlap = be.iou(p, g)
    pc = ((p[0] + p[2]) / 2, (p[1] + p[3]) / 2)
    gc = ((g[0] + g[2]) / 2, (g[1] + g[3]) / 2)
    area_p = (p[2] - p[0]) * (p[3] - p[1])
    area_g = (g[2] - g[0]) * (g[3] - g[1])
    size_ratio = min(area_p, area_g) / max(area_p, area_g)
    near = max(0.0, 1.0 - math.dist(pc, gc) / 0.25) * math.sqrt(size_ratio)
    return 0.7 * overlap + 0.3 * near


def detection_score(preds, targets):
    if not preds or not targets:
        return 0.0
    candidates = sorted(
        ((pair_quality(pred, gt), i, j) for i, pred in enumerate(preds) for j, gt in enumerate(targets)),
        reverse=True,
    )
    used_p, used_g, quality = set(), set(), 0.0
    for score, i, j in candidates:
        if score <= 0:
            break
        if i not in used_p and j not in used_g:
            quality += score
            used_p.add(i)
            used_g.add(j)
    soft_f1 = 2 * quality / (len(preds) + len(targets))
    tp, _, _ = be.match(preds, targets, class_aware=True, threshold=0.5)
    hard_f1 = 2 * tp / (len(preds) + len(targets))
    return 0.75 * soft_f1 + 0.25 * hard_f1


def format_reward(completions, **kwargs):
    return [1.0 if (boxes := strict_boxes(completion_text(c))) is not None and boxes else 0.0 for c in completions]


audit_path = Path(OUTPUT_DIR) / "rollout_samples.jsonl"
reward_calls = 0


def detection_reward(completions, gt_boxes, item_id=None, **kwargs):
    global reward_calls
    reward_calls += 1
    scores = []
    predicted_counts = []
    for completion, targets_json in zip(completions, gt_boxes, strict=True):
        targets = json.loads(targets_json)
        preds = strict_boxes(completion_text(completion))
        predicted_counts.append(len(preds) if preds is not None else 0)
        scores.append(detection_score(preds, targets) if preds is not None else 0.0)
    if log_metric := kwargs.get("log_metric"):
        log_metric("audit/predicted_boxes_mean", sum(predicted_counts) / len(predicted_counts))
        log_metric("audit/nonzero_detection_reward_rate", sum(value > 0 for value in scores) / len(scores))
    if reward_calls == 1 or reward_calls % 10 == 0:
        audit_path.parent.mkdir(parents=True, exist_ok=True)
        step = getattr(kwargs.get("trainer_state"), "global_step", None)
        with audit_path.open("a") as stream:
            for i in range(min(4, len(completions))):
                stream.write(json.dumps({
                    "step": step, "item_id": item_id[i] if item_id is not None else None,
                    "completion": completion_text(completions[i]), "gt_boxes": json.loads(gt_boxes[i]),
                    "detection_reward": scores[i],
                }) + "\n")
    return scores

In [ ]:
def chat(text):
    return [{"role": "assistant", "content": text}]

target = [["erosion", [0.4, 0.4, 0.6, 0.6]]]
perfect = chat('[{"label":"erosion","bbox_2d":[400,400,600,600]}]')
near = chat('[{"label":"erosion","bbox_2d":[550,400,750,600]}]')
far = chat('[{"label":"erosion","bbox_2d":[800,800,900,900]}]')
wrong = chat('[{"label":"paint cracks","bbox_2d":[400,400,600,600]}]')
duplicate = chat('[{"label":"erosion","bbox_2d":[400,400,600,600]},{"label":"erosion","bbox_2d":[400,400,600,600]}]')
huge = chat('[{"label":"erosion","bbox_2d":[0,0,1000,1000]}]')
extra = chat('[{"label":"erosion","bbox_2d":[400,400,600,600]},{"label":"erosion","bbox_2d":[800,800,900,900]}]')
score = lambda c: detection_score(strict_boxes(completion_text(c)), target)
assert score(perfect) > score(near) > score(far) == 0
assert score(perfect) > score(duplicate) > 0
assert score(near) > score(huge) > 0
assert score(perfect) > score(extra) > 0
assert strict_boxes('[{"label":"erosion","bbox_2d":[-1,0,100,100]}]') is None
assert score(wrong) == 0 and score(chat("[]")) == 0
assert format_reward([perfect, chat("[]"), chat("not JSON")]) == [1.0, 0.0, 0.0]
print({name: round(score(c), 3) for name, c in [("perfect", perfect), ("near", near), ("far", far), ("wrong type", wrong), ("duplicate", duplicate)]})

## Load the SFT adapter

The pretrained LoRA is trainable. With TRL 1.16 and PEFT 0.21.2, a nonzero KL coefficient makes a frozen reference copy of that starting adapter. Do not pass a new `peft_config` to `GRPOTrainer` alongside this pretrained `PeftModel`.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from transformers import set_seed

set_seed(SEED)
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=base_info["revision"], trust_remote_code=True)
processor.tokenizer.padding_side = "left"
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, revision=base_info["revision"], dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
model = PeftModel.from_pretrained(
    base_model, SFT_ADAPTER_REPO, revision=start_info["revision"], is_trainable=True,
)
model.print_trainable_parameters()
assert any(parameter.requires_grad for parameter in model.parameters())
print("SFT adapter:", start_info)

## Baseline on the validation monitor

Before training, score the 200 validation photos with the SFT adapter. The same photos are scored at every checkpoint and after training. Each check takes a few minutes and prints detection F1, precision, recall, box-only F1, type accuracy, parse rate and detection counts. Small changes on 200 photos are noisy, so look for a clear trend, and pick a checkpoint on all 720 validation photos.

In [ ]:
monitor_val_dir = Path(OUTPUT_DIR) / "monitor_val"
validation_log_path = Path(OUTPUT_DIR) / "validation_log.jsonl"
monitor_config = {
    "base_model": base_info,
    "adapter": start_info,
    "split": "val-monitor-200",
    "dataset_url": be.DATASET_URL,
    "exclude_cross_class_duplicates": True,
    "max_image_tokens": None,
}


def record_validation(step, phase, records, result_path, reference_f1=None):
    assert len(records) == len(val_items), f"Incomplete validation: {len(records)}/{len(val_items)} images"
    metrics = be.summarize(records).iloc[0]
    row = {
        "step": int(step), "phase": phase, "n_images": len(records),
        "f1": float(metrics["f1"]), "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]), "f1_box_only": float(metrics["f1_box_only"]),
        "type_accuracy": float(metrics["type_accuracy"]),
        "parse_rate": float(metrics["parse_rate"]),
        "tp": sum(r["tp"] for r in records), "fp": sum(r["fp"] for r in records),
        "fn": sum(r["fn"] for r in records), "result_path": str(result_path),
    }
    if reference_f1 is not None:
        row["delta_f1_vs_sft"] = row["f1"] - reference_f1
    validation_log_path.parent.mkdir(parents=True, exist_ok=True)
    with validation_log_path.open("a") as stream:
        stream.write(json.dumps(row) + "\n")
    delta = f" delta_vs_sft={row['delta_f1_vs_sft']:+.3f}" if reference_f1 is not None else ""
    print(
        f"[validation] step={step} phase={phase} n={row['n_images']} "
        f"f1={row['f1']:.3f}{delta} precision={row['precision']:.3f} "
        f"recall={row['recall']:.3f} box_f1={row['f1_box_only']:.3f} "
        f"type_acc={row['type_accuracy']:.3f} parse={row['parse_rate']:.3f} "
        f"tp={row['tp']} fp={row['fp']} fn={row['fn']}", flush=True,
    )
    return row


model.eval()
before_path = be.run_local(model, processor, val_items, "sft-before-grpo", BOX_FORMAT,
                           results_dir=monitor_val_dir, run_config=monitor_config)
before_records = be.read_jsonl(before_path)
baseline_row = record_validation(0, "sft", before_records, before_path)
baseline_f1 = baseline_row["f1"]


## Train

The trainer saves an adapter every `SAVE_STEPS` steps and scores it on the 200 validation photos. To stop early, interrupt the cell: the current adapter is saved, and you can run the remaining cells to score it and download the results. A disconnected Colab runtime can't save anything, so keep it active until the download finishes.

Health metrics are saved every step and printed every 10 steps. Watch the reward components, `reward_std` and `frac_reward_zero_std` (groups where all four answers scored the same, which teach nothing), KL, truncated completions, gradient norm and memory. `rollout_samples.jsonl` keeps sampled answers with their rewards. A small drop on one 200-photo check isn't a reason to stop; look for a sustained change or a clear training-health problem.

In [ ]:
from datetime import datetime, timezone
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer
import time


class HealthLog(TrainerCallback):
    def __init__(self, path):
        self.path = Path(path)
        self.started = time.monotonic()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero or not logs:
            return control
        row = {"time_utc": datetime.now(timezone.utc).isoformat(), "step": state.global_step,
               "elapsed_minutes": round((time.monotonic() - self.started) / 60, 2), **logs,
               "gpu_allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 2),
               "gpu_peak_allocated_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2)}
        self.path.parent.mkdir(parents=True, exist_ok=True)
        safe = {key: str(value) if isinstance(value, float) and not math.isfinite(value) else value for key, value in row.items()}
        with self.path.open("a") as stream:
            stream.write(json.dumps(safe, default=str, allow_nan=False) + "\n")
        keys = ("reward", "rewards/format_reward/mean", "rewards/detection_reward/mean", "reward_std", "frac_reward_zero_std", "kl", "completions/clipped_ratio", "audit/predicted_boxes_mean", "grad_norm")
        detail = " ".join(f"{key}={row[key]:.3g}" for key in keys if isinstance(row.get(key), (int, float)))
        if state.global_step == 1 or state.global_step % 10 == 0:
            print(f"[grpo] step={state.global_step} {detail} gpu={row['gpu_allocated_gb']:.2f}GB", flush=True)
        if any(isinstance(row.get(key), (int, float)) and not math.isfinite(row[key]) for key in ("loss", "reward", "grad_norm")):
            print("WARNING: non-finite GRPO metric; stopping training.")
            control.should_training_stop = True
        return control


class DetectionMonitor(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return control
        step = int(state.global_step)
        if step >= args.max_steps:
            return control  # the final adapter is scored after training
        checkpoint = Path(args.output_dir) / f"checkpoint-{step}"
        checkpoint_info = be.model_identity(str(checkpoint))
        current_model = kwargs["model"]
        was_training = current_model.training
        try:
            current_model.eval()
            run_config = {**monitor_config, "adapter": checkpoint_info}
            result_path = be.run_local(
                current_model, processor, val_items, f"grpo-step-{step}", BOX_FORMAT,
                results_dir=monitor_val_dir, run_config=run_config,
            )
            records = be.read_jsonl(result_path)
            record_validation(step, "checkpoint", records, result_path, baseline_f1)
        finally:
            current_model.train(was_training)
        return control


config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=2,
    lr_scheduler_type="linear",
    bf16=True,
    gradient_checkpointing=True,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,  # one prompt group of four completions per optimizer step
    max_completion_length=MAX_COMPLETION_LENGTH,
    mask_truncated_completions=True,
    temperature=1.0,
    beta=0.01,
    disable_dropout=True,
    reward_weights=[0.02, 1.0],
    chat_template_kwargs={"enable_thinking": False},
    remove_unused_columns=False,
    logging_steps=1,
    logging_nan_inf_filter=False,
    log_completions=True,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=8,
    report_to="none",
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, detection_reward],
    args=config,
    train_dataset=train_ds,
    processing_class=processor,
    callbacks=[HealthLog(Path(OUTPUT_DIR) / "training_log.jsonl"), DetectionMonitor()],
)
try:
    trainer.train()
except KeyboardInterrupt:
    print(f"Stopped by user at step {trainer.state.global_step}; saving the current adapter.", flush=True)
finally:
    trainer.save_model(OUTPUT_DIR)
    print("Saved GRPO adapter to", OUTPUT_DIR, "at step", trainer.state.global_step, flush=True)

## Score the trained adapter

The final adapter is scored on the same 200 validation photos. The validation log then holds the SFT baseline, every checkpoint and the final adapter. To pick a checkpoint, score the candidates on all 720 validation photos.


In [ ]:
model = trainer.model.eval()
trained_adapter = be.model_identity(OUTPUT_DIR)
after_config = {**monitor_config, "adapter": trained_adapter}
after_path = be.run_local(model, processor, val_items, "grpo-after", BOX_FORMAT, results_dir=monitor_val_dir, run_config=after_config)
after_records = be.read_jsonl(after_path)
after_row = record_validation(trainer.state.global_step, "final", after_records, after_path, baseline_f1)

## Inspect logs and sampled completions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

health = pd.DataFrame(json.loads(line) for line in (Path(OUTPUT_DIR) / "training_log.jsonl").read_text().splitlines())
health.to_csv(Path(OUTPUT_DIR) / "training_log.csv", index=False)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, columns, title in [
    (axes[0,0], ["reward", "rewards/format_reward/mean", "rewards/detection_reward/mean"], "Rewards"),
    (axes[0,1], ["reward_std", "frac_reward_zero_std", "audit/nonzero_detection_reward_rate"], "Group signal"),
    (axes[1,0], ["kl", "grad_norm"], "KL and gradient"),
    (axes[1,1], ["completions/clipped_ratio", "audit/predicted_boxes_mean", "gpu_allocated_gb"], "Truncation and memory"),
]:
    for col in columns:
        if col in health:
            values = pd.to_numeric(health[col], errors="coerce")
            ax.plot(health["step"], values, label=col)
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR) / "training_health.png", dpi=150)
plt.show()
if audit_path.exists():
    samples = [json.loads(line) for line in audit_path.read_text().splitlines()]
    print(f"Saved {len(samples)} audited rollouts. Latest examples:")
    for sample in samples[-4:]:
        print(sample["item_id"], "reward=", round(sample["detection_reward"], 3),
              "completion=", sample["completion"][:500])
validation = pd.DataFrame(json.loads(line) for line in validation_log_path.read_text().splitlines())
validation.to_csv(Path(OUTPUT_DIR) / "validation_log.csv", index=False)
print(validation[["step", "phase", "f1", "delta_f1_vs_sft", "precision", "recall", "f1_box_only", "type_accuracy", "parse_rate", "tp", "fp", "fn"]].to_string(index=False))


## Download adapters, validation results and logs

The ZIP holds the final adapter, every saved checkpoint (without optimizer states), the training and validation logs, the sampled answers, the plots and the validation predictions. Download it before the Colab runtime ends.

In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_STORED

output = Path(OUTPUT_DIR)
archive_out = Path(f"{output.name}-evidence.zip")
with ZipFile(archive_out, "w", compression=ZIP_STORED) as bundle:
    for folder in [output, *sorted(output.glob("checkpoint-*"))]:
        for file in [*sorted(folder.glob("adapter_model*.safetensors")), *sorted(folder.glob("adapter_config.json"))]:
            bundle.write(file, arcname=file.relative_to(output.parent))
    for pattern in ("training_log.*", "validation_log.*", "training_health.png", "rollout_samples.jsonl", "monitor_val/*.jsonl"):
        for file in output.glob(pattern):
            bundle.write(file, arcname=file.relative_to(output.parent))
with ZipFile(archive_out) as check:
    assert check.testzip() is None
print(archive_out, archive_out.stat().st_size / 1024**2, "MiB")
files.download(str(archive_out))

## Push the adapter

Push the final adapter to `HUB_REPO`, so you can load it in [`evaluate.ipynb`](evaluate.ipynb) or anywhere else on top of `LiquidAI/LFM3-VL-2B`.

In [ ]:
if HUB_REPO:
    trainer.model.push_to_hub(HUB_REPO, private=True)
    print(f"Pushed the adapter to https://huggingface.co/{HUB_REPO}")
else:
    print(f"HUB_REPO not set; the adapter is only in {OUTPUT_DIR}. Download the ZIP before the Colab runtime ends.")

## Next steps

- **Pick a checkpoint:** score the saved checkpoints and the SFT starting point on all 720 validation photos. A rising training reward alone doesn't mean better detection.
- **Test once, at the end:** set `RUN = "lfm3_finetune"` and `LFM3_ADAPTER_ID = HUB_REPO` in [`evaluate.ipynb`](evaluate.ipynb), and compare the fine-tuned model with the base model and frontier models on the held-out test split.
- **Change one thing at a time:** keep the prompt, box format and split from `blade_eval.py` fixed, so results stay comparable.